# ParaLoRA fine-tuning (sequence branch)

Trains the LoRA-augmented ProtT5 classifier on the PECAN paratope task.
This notebook reproduces the released checkpoint using the same JSON
configuration as the CLI entrypoint.

Required artefacts (see README.md for the data preparation pipeline):

* `configs/paralora.json` -- hyper-parameters
* `data/paralora/train.csv`, `val.csv`, `test.csv` -- canonical sequence splits
* `runs/paralora/` -- output directory (created automatically)


In [ ]:
import os, json, torch
from paralora.data import load_split, prepare_split
from paralora.trainer import train_per_residue, set_seeds
from paralora.evaluate import evaluate_paratope


In [ ]:
with open('configs/paralora.json') as f:
    cfg = json.load(f)
# Point this at a local copy of the ProtT5 weights when not using HF cache.
cfg['model_name_or_path'] = os.environ.get('PROTT5_PATH', cfg['model_name_or_path'])
print('Loaded config:', cfg)


In [ ]:
train_split = load_split('data/paralora/train.csv')
val_split = load_split('data/paralora/val.csv')
print('train rows:', len(train_split.sequences), 'val rows:', len(val_split.sequences))


In [ ]:
tokenizer, model, history, _ = train_per_residue(
    train_split=train_split,
    valid_split=val_split,
    config=cfg,
    output_dir='runs/paralora',
    deepspeed=True,
)
print('Final epoch log:', history[-1])


In [ ]:
result = evaluate_paratope(
    config=cfg,
    checkpoint_path='runs/paralora/trainable_params.pt',
    split_path='data/paralora/test.csv',
    batch_size=16,
    output_path='runs/paralora/eval.json',
)
print('Test metrics:', result.metrics)
